# =============================================
# 📊 DOW JONES MARKET INTELLIGENCE NOTEBOOK
# =============================================

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/dow-jones-global-market-shock-dataset/DJIA.csv


# =============================================
# 1. IMPORT LIBRARIES
# =============================================

In [16]:
import plotly.io as pio

pio.renderers.default = 'iframe'

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

plt.style.use('dark_background')

# =============================================
# 2. LOAD DATA
# =============================================

In [3]:

# Replace with your dataset path
file_path = "/kaggle/input/datasets/krupalpatel07/dow-jones-global-market-shock-dataset/DJIA.csv"
df = pd.read_csv(file_path)

# =============================================
# 3. PREPROCESSING
# =============================================

In [5]:

df['date'] = pd.to_datetime(df['Date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

# =============================================
# 4. HEADERS
# =============================================

In [6]:
from IPython.display import display, HTML

def neon_header(text):
    display(HTML(f"""
    <div style="
        background: linear-gradient(90deg, #0f2027, #2c5364);
        padding: 15px;
        border-radius: 10px;
        box-shadow: 0 0 20px #00f2ff;
        margin-top:20px;
    ">
        <h1 style="color:#00f2ff; text-align:center;">{text}</h1>
    </div>
    """))

neon_header("🚀 Dow Jones Price Evolution")

# =============================================
# 5. PRICE CHART (INTERACTIVE)
# =============================================

In [17]:

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close'],
    name='Market'
))

fig.update_layout(title='Dow Jones Candlestick', template='plotly_dark')
fig.show()


# =============================================
# 6. TECHNICAL INDICATORS
# =============================================

In [9]:
neon_header("📈 Technical Indicators")

# Moving Averages
df['MA50'] = df['Close'].rolling(50).mean()
df['MA200'] = df['Close'].rolling(200).mean()

# RSI
window = 14
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='Close'))
fig.add_trace(go.Scatter(x=df.index, y=df['MA50'], name='MA50'))
fig.add_trace(go.Scatter(x=df.index, y=df['MA200'], name='MA200'))
fig.update_layout(template='plotly_dark')
fig.show()


# =============================================
# 7. VOLATILITY REGIME DETECTION
# =============================================

In [11]:
neon_header("⚡ Volatility Regime Detection")

# Returns
df['returns'] = df['Close'].pct_change()

# Rolling volatility
df['volatility'] = df['returns'].rolling(20).std()

# Regime classification
df['regime'] = np.where(df['volatility'] > df['volatility'].median(), 'High Vol', 'Low Vol')

fig = px.scatter(df, x=df.index, y='volatility', color='regime', template='plotly_dark')
fig.show()


# =============================================
# 8. MACHINE LEARNING MODEL
# =============================================

In [13]:
neon_header("🤖 ML Price Prediction (Random Forest)")

# Feature Engineering
df['lag1'] = df['Close'].shift(1)
df['lag2'] = df['Close'].shift(2)
df['lag3'] = df['Close'].shift(3)

df = df.dropna()

X = df[['lag1','lag2','lag3']]
y = df['Close']

# Train-test split
split = int(len(df)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Model
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

# Predictions
preds = model.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"RMSE: {rmse}")

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test.index, y=y_test, name='Actual'))
fig.add_trace(go.Scatter(x=y_test.index, y=preds, name='Predicted'))
fig.update_layout(template='plotly_dark')
fig.show()

RMSE: 10660.157700987289


# =============================================
# 9. ADVANCED VISUAL - HEATMAP CORRELATION
# =============================================

In [14]:

neon_header("🔥 Feature Correlation Matrix")

corr = df[['Close','MA50','MA200','RSI','volatility']].corr()

fig = px.imshow(corr, text_auto=True, template='plotly_dark')
fig.show()

# =============================================
# 10. FINAL INSIGHTS
# =============================================

In [15]:
neon_header("📌 Final Insights")

print("""
1. Market shows clear volatility clustering.
2. Moving averages highlight long-term trend.
3. ML captures short-term structure but not macro shocks.
4. High volatility regime aligns with major price swings.
""")



1. Market shows clear volatility clustering.
2. Moving averages highlight long-term trend.
3. ML captures short-term structure but not macro shocks.
4. High volatility regime aligns with major price swings.




# =============================================
# END OF NOTEBOOK
# =============================================
